
# V35 LIVE — Cross-Script Banglish Rescue

**Public leaderboard champion: V14 = 0.586**  
**New target: ≥ 0.595**

V35 targets the largest remaining distribution-specific weakness instead of replacing the whole system.

## Why cross-script?

On the LIVE validation set:

- native Bengali dominates the data and V14 is already fairly strong there;
- Romanized/Banglish is ~18–20% of validation/test;
- Romanized text has a much higher toxicity prevalence;
- V14's Romanized `is_toxic` score is much weaker than its native-Bengali score;
- V14 misses Romanized implicit-toxicity cases entirely.

## New signals

### 1. Banglish cross-script E5 embeddings
`kazalbrur/bangla-embed-e5-small-banglish`

A compact cross-script sentence encoder trained to align:
- Bangla script
- Banglish / Romanized Bangla
- mixed text

Frozen embeddings + 5-fold logistic OOF classifiers.

### 2. BGE-M3 Bangla cross-script embeddings
`nahidstaq/bge-m3-bangla`

A larger cross-script semantic embedding model. Frozen embeddings + 5-fold logistic OOF classifiers.

### 3. Transliteration-pretrained XLM-R
`aplycaebous/tb-XLM-R-fpt`

XLM-R further pretrained on transliterated Bangla. We fine-tune it with script-aware sampling.

## Safety strategy

- Native Bengali predictions remain V14 by default.
- New models are allowed to change **Roman or mixed-script rows only**.
- Candidate counts are anchored to train/validation script-specific prevalence.
- A script/label replacement must improve both:
  1. its script-specific validation score, and
  2. the full label F1.
- The notebook tracks validation-fold stability and total test changes.
- Exact accepted Kaggle IDs are enforced.

Send `v35_results.zip` to ChatGPT before submitting.


In [ ]:

# 1. Upload V35 LIVE data
from google.colab import files
import pathlib,zipfile

uploaded=files.upload()
assert "v35_LIVE_crossscript_data.zip" in uploaded

DATA_DIR=pathlib.Path("/content/v35_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v35_LIVE_crossscript_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ",p.name,f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install + model preflight
import sys,subprocess

subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "sentence-transformers>=3.0.0",
    "scikit-learn",
    "scipy",
    "pandas",
    "numpy",
    "tqdm",
    "safetensors"
])

from transformers import AutoConfig,AutoTokenizer
from sentence_transformers import SentenceTransformer

TB_MODEL="aplycaebous/tb-XLM-R-fpt"
E5_MODEL="kazalbrur/bangla-embed-e5-small-banglish"
BGE_MODEL="nahidstaq/bge-m3-bangla"

print("Checking transliterated XLM-R...")
_=AutoConfig.from_pretrained(TB_MODEL)
_=AutoTokenizer.from_pretrained(TB_MODEL)

print("Checking E5 Banglish...")
_=SentenceTransformer(E5_MODEL,device="cpu")

print("Checking BGE-M3 Bangla...")
_=SentenceTransformer(BGE_MODEL,device="cpu")

print("✅ all V35 checkpoints accessible")


In [ ]:

# 3. Imports, strict LIVE verification, script audit
import gc,re,math,json,random,unicodedata,zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

SEED=2026
N_FOLDS=5
EPOCHS=3
SNAPSHOT_EPOCHS=[2,3]
MAX_LENGTH=128
BATCH=4
ACCUM=4
EVAL_BATCH=24
LR=1.2e-5
HEAD_LR=2.5e-5
WEIGHT_DECAY=.01
WARMUP_RATIO=.10

TB_MODEL="aplycaebous/tb-XLM-R-fpt"
E5_MODEL="kazalbrur/bangla-embed-e5-small-banglish"
BGE_MODEL="nahidstaq/bge-m3-bangla"

LABELS=[
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]
LI={l:i for i,l in enumerate(LABELS)}

RUN_DIR=Path("/content/v35_output")
RUN_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

seed_all(SEED)
assert torch.cuda.is_available(),"Enable a T4 GPU."

train=pd.read_csv(DATA_DIR/"train.csv",dtype={"ID":str})
valid=pd.read_csv(DATA_DIR/"validation.csv",dtype={"ID":str})
test=pd.read_csv(DATA_DIR/"test.csv",dtype={"ID":str})
champ=pd.read_csv(DATA_DIR/"submission_v14.csv",dtype={"ID":str})
v2v=pd.read_csv(DATA_DIR/"v2_valid.csv",dtype={"ID":str})
v2t=pd.read_csv(DATA_DIR/"v2_test.csv",dtype={"ID":str})

assert len(train)==3297 and len(valid)==500 and len(test)==1000
assert test.ID.tolist()==champ.ID.tolist()
assert test.ID.head(4).tolist()==[
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]

BN_RE=re.compile(r"[\u0980-\u09FF]")
LAT_RE=re.compile(r"[A-Za-z]")

def clean(x):
    x=unicodedata.normalize("NFKC",str(x))
    x=x.replace("\u200c"," ").replace("\u200d"," ")
    return re.sub(r"\s+"," ",x).strip()

def script_type(x):
    s=str(x)
    bn=bool(BN_RE.search(s))
    lat=bool(LAT_RE.search(s))

    if bn and lat:
        return "mixed"
    if bn:
        return "native"
    return "roman"

def group_key(x):
    return clean(x).lower()

for df in [train,valid,test]:
    df["clean_text"]=df.text.map(clean)
    df["script"]=df.text.map(script_type)
    df["group_key"]=df.text.map(group_key)

combined=pd.concat([train,valid],ignore_index=True)

print("✅ exact LIVE Kaggle IDs verified")
print("\nScript counts:")
display(pd.DataFrame({
    "train":train.script.value_counts(),
    "validation":valid.script.value_counts(),
    "test":test.script.value_counts(),
}).fillna(0).astype(int))

print("\nLIVE label prevalence by script:")
display(
    train.groupby("script")[LABELS].mean().round(4)
)


In [ ]:

# 4. Recreate grouped 5-fold split on LIVE combined data
groups=combined.groupby("group_key",sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in LABELS}
).reset_index()

global_pos=np.array([combined[l].sum() for l in LABELS],float)
total=float(len(combined))
rarity=1/np.sqrt(np.maximum(global_pos,1))
mat=groups[[f"sum_{l}" for l in LABELS]].to_numpy(float)

rng=np.random.default_rng(SEED)
priority=(mat*rarity).sum(1)+.005*groups["size"].to_numpy()
order=np.argsort(-(priority+rng.random(len(groups))*1e-8))

fp=np.zeros((5,len(LABELS)))
fs=np.zeros(5)
assign={}

for idx in order:
    row=groups.iloc[idx]
    vec=np.array([row[f"sum_{l}"] for l in LABELS],float)
    size=float(row["size"])

    costs=[]
    for f in range(5):
        p=fp.copy()
        s=fs.copy()

        p[f]+=vec
        s[f]+=size

        costs.append(
            np.mean(
                (p/np.maximum(global_pos[None,:],1)-.2)**2
            )
            +.30*np.mean((s/total-.2)**2)
        )

    best=int(np.argmin(costs))
    assign[row.group_key]=best
    fp[best]+=vec
    fs[best]+=size

combined["fold"]=combined.group_key.map(assign).astype(int)

assert combined.groupby("group_key").fold.nunique().max()==1

print("Fold sizes:",combined.groupby("fold").size().to_dict())
display(combined.groupby(["fold","script"]).size().unstack(fill_value=0))


In [ ]:

# 5. Extract frozen cross-script embeddings

all_texts=combined.clean_text.tolist()
test_texts=test.clean_text.tolist()

def encode_model(model_name,out_prefix,batch_size):
    cache_oof=RUN_DIR/f"{out_prefix}_combined_embeddings.npy"
    cache_test=RUN_DIR/f"{out_prefix}_test_embeddings.npy"

    if cache_oof.exists() and cache_test.exists():
        print("✅ reuse embeddings:",out_prefix)
        return np.load(cache_oof),np.load(cache_test)

    print("Loading:",model_name)
    model=SentenceTransformer(
        model_name,
        device="cuda"
    )

    emb_all=model.encode(
        all_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    emb_test=model.encode(
        test_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    np.save(cache_oof,emb_all)
    np.save(cache_test,emb_test)

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return emb_all,emb_test

e5_emb,e5_test_emb=encode_model(
    E5_MODEL,"e5_banglish",32
)

bge_emb,bge_test_emb=encode_model(
    BGE_MODEL,"bge_m3_bangla",12
)

print("Embedding shapes:")
print("E5:",e5_emb.shape,e5_test_emb.shape)
print("BGE:",bge_emb.shape,bge_test_emb.shape)


In [ ]:

# 6. 5-fold OOF classifiers on frozen embeddings

def embedding_oof(emb,test_emb,prefix):
    oof=np.zeros((len(combined),len(LABELS)),np.float32)
    test_parts=[]

    for fold in range(5):
        tr=combined.fold.to_numpy()!=fold
        va=~tr

        pv=np.zeros((va.sum(),len(LABELS)),np.float32)
        pt=np.zeros((len(test),len(LABELS)),np.float32)

        for j,label in enumerate(LABELS):
            y=combined[label].to_numpy(int)

            # Mild positive weighting, not balanced weighting.
            pos=y[tr].sum()
            neg=tr.sum()-pos
            boost=min(
                3.0,
                max(1.0,float(np.sqrt(neg/max(pos,1))))
            )

            sw=np.ones(tr.sum(),dtype=float)
            sw[y[tr]==1]=boost

            clf=LogisticRegression(
                C=.35,
                solver="liblinear",
                max_iter=1200,
                random_state=SEED+fold,
            )
            clf.fit(
                emb[tr],
                y[tr],
                sample_weight=sw
            )

            pv[:,j]=clf.predict_proba(
                emb[va]
            )[:,1]

            pt[:,j]=clf.predict_proba(
                test_emb
            )[:,1]

        oof[va]=pv
        test_parts.append(pt)

        print(prefix,"fold",fold,"done")

    ptest=np.mean(test_parts,axis=0)

    np.savez_compressed(
        RUN_DIR/f"{prefix}_oof_test.npz",
        oof=oof,
        test=ptest
    )

    return oof,ptest

e5_oof,e5_test=embedding_oof(
    e5_emb,e5_test_emb,"e5_banglish"
)

bge_oof,bge_test=embedding_oof(
    bge_emb,bge_test_emb,"bge_bangla"
)

print("✅ frozen embedding OOF complete")


In [ ]:

# 7. Fine-tune transliteration-pretrained XLM-R with script-aware sampling

class DS(Dataset):
    def __init__(self,df,tok,labels=True):
        self.x=df.clean_text.astype(str).tolist()
        self.y=df[LABELS].to_numpy(np.float32) if labels else None
        self.tok=tok

    def __len__(self):
        return len(self.x)

    def __getitem__(self,i):
        e=self.tok(
            self.x[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        if self.y is not None:
            e["labels"]=self.y[i]
        return e

class Coll:
    def __init__(self,tok):
        self.tok=tok

    def __call__(self,items):
        y=None

        if "labels" in items[0]:
            y=torch.tensor(
                np.stack([x.pop("labels") for x in items]),
                dtype=torch.float32
            )

        b=self.tok.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt"
        )

        if y is not None:
            b["labels"]=y

        return b

def competition_loss(logits,y):
    raw=torch.nn.functional.binary_cross_entropy_with_logits(
        logits.float(),
        y,
        reduction="none"
    )

    task=torch.tensor(
        [.50]+[.50/6]*6,
        dtype=torch.float32,
        device=logits.device
    ).unsqueeze(0)

    # Mild positive boost only.
    posboost=torch.tensor(
        [1.0,1.5,1.2,1.2,1.6,1.1,1.5],
        dtype=torch.float32,
        device=logits.device
    ).unsqueeze(0)

    w=torch.where(
        y>.5,
        posboost,
        torch.ones_like(y)
    )

    return (raw*w*task).sum(1).mean()

@torch.no_grad()
def predict(model,loader):
    model.eval()
    parts=[]

    for b in loader:
        b.pop("labels",None)

        inp={
            k:v.to("cuda",non_blocking=True)
            for k,v in b.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits=model(**inp).logits

        parts.append(
            torch.sigmoid(
                logits.float()
            ).cpu().numpy()
        )

    return np.concatenate(parts)

def train_tb_xlmr():
    tok=AutoTokenizer.from_pretrained(TB_MODEL)

    oof=np.zeros(
        (len(combined),len(LABELS)),
        np.float32
    )
    test_parts=[]
    history=[]

    fold_seeds=[2026,17,3407,1337,42]

    for fold in range(5):
        tr=combined[
            combined.fold!=fold
        ].copy()

        va=combined[
            combined.fold==fold
        ].copy()

        cache=RUN_DIR/f"tbxlmr_fold_{fold}.npz"
        hp=RUN_DIR/f"tbxlmr_fold_{fold}_history.csv"

        if cache.exists():
            with np.load(cache) as z:
                idx=z["idx"]
                pv=z["valid"]
                pt=z["test"]

            if np.array_equal(
                idx,
                va.index.to_numpy()
            ):
                print("✅ reuse TB-XLM-R fold",fold)
                oof[idx]=pv
                test_parts.append(pt)
                continue

        seed_all(fold_seeds[fold])

        # Script-aware oversampling.
        script_w=tr.script.map({
            "native":.65,
            "mixed":1.60,
            "roman":2.20
        }).to_numpy(float)

        # Additional emphasis on rare semantic labels.
        target_w=(
            1.0
            +1.5*tr.implicit_toxicity.to_numpy()
            +.50*tr.sarcasm.to_numpy()
            +1.0*tr.threatening.to_numpy()
            +1.0*tr.sexual_content.to_numpy()
        )

        sampler=WeightedRandomSampler(
            torch.tensor(
                script_w*target_w,
                dtype=torch.double
            ),
            num_samples=len(tr),
            replacement=True,
        )

        gc.collect()
        torch.cuda.empty_cache()

        col=Coll(tok)

        tl=DataLoader(
            DS(tr,tok,True),
            batch_size=BATCH,
            sampler=sampler,
            num_workers=0,
            pin_memory=True,
            collate_fn=col
        )

        vl=DataLoader(
            DS(va,tok,True),
            batch_size=EVAL_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=col
        )

        tel=DataLoader(
            DS(test,tok,False),
            batch_size=EVAL_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=col
        )

        model=AutoModelForSequenceClassification.from_pretrained(
            TB_MODEL,
            num_labels=len(LABELS),
            problem_type="multi_label_classification",
            ignore_mismatched_sizes=True,
        ).to("cuda")

        # Separate head/backbone LR.
        head=[]
        backbone=[]

        for n,p in model.named_parameters():
            if (
                "classifier" in n
                or "score" in n
            ):
                head.append(p)
            else:
                backbone.append(p)

        opt=torch.optim.AdamW([
            {
                "params":backbone,
                "lr":LR,
                "weight_decay":WEIGHT_DECAY
            },
            {
                "params":head,
                "lr":HEAD_LR,
                "weight_decay":WEIGHT_DECAY
            }
        ])

        steps=math.ceil(
            len(tl)/ACCUM
        )*EPOCHS

        sched=get_linear_schedule_with_warmup(
            opt,
            int(steps*WARMUP_RATIO),
            steps
        )

        scaler=torch.amp.GradScaler("cuda")

        sv=[]
        st=[]
        fh=[]

        for epoch in range(1,EPOCHS+1):
            model.train()
            opt.zero_grad(set_to_none=True)

            running=0.
            seen=0

            for step,b in enumerate(
                tqdm(
                    tl,
                    desc=(
                        f"TB-XLM-R f{fold} "
                        f"e{epoch}/{EPOCHS}"
                    )
                )
            ):
                y=b.pop("labels").to(
                    "cuda",
                    non_blocking=True
                )

                inp={
                    k:v.to(
                        "cuda",
                        non_blocking=True
                    )
                    for k,v in b.items()
                }

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):
                    logits=model(**inp).logits

                loss=competition_loss(
                    logits,y
                )

                if not torch.isfinite(loss):
                    raise FloatingPointError(
                        "non-finite TB-XLM-R loss"
                    )

                scaler.scale(
                    loss/ACCUM
                ).backward()

                running+=float(loss.item())*len(y)
                seen+=len(y)

                if (
                    (step+1)%ACCUM==0
                    or step+1==len(tl)
                ):
                    scaler.unscale_(opt)

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),1.0
                    )

                    old=scaler.get_scale()
                    scaler.step(opt)
                    scaler.update()

                    if scaler.get_scale()>=old:
                        sched.step()

                    opt.zero_grad(
                        set_to_none=True
                    )

            rec={
                "fold":fold,
                "epoch":epoch,
                "loss":running/max(seen,1)
            }

            print(rec)
            fh.append(rec)
            history.append(rec)

            pd.DataFrame(fh).to_csv(
                hp,index=False
            )

            if epoch in SNAPSHOT_EPOCHS:
                sv.append(
                    predict(model,vl)
                )
                st.append(
                    predict(model,tel)
                )

        pv=np.mean(sv,axis=0)
        pt=np.mean(st,axis=0)

        oof[va.index.to_numpy()]=pv
        test_parts.append(pt)

        np.savez_compressed(
            cache,
            idx=va.index.to_numpy(),
            valid=pv,
            test=pt
        )

        del model,opt,sched,scaler
        del tl,vl,tel,sampler

        gc.collect()
        torch.cuda.empty_cache()

    ptest=np.mean(
        test_parts,
        axis=0
    )

    np.savez_compressed(
        RUN_DIR/"tbxlmr_oof_test.npz",
        oof=oof,
        test=ptest
    )

    pd.DataFrame(history).to_csv(
        RUN_DIR/"tbxlmr_training_history.csv",
        index=False
    )

    return oof,ptest

tb_oof,tb_test=train_tb_xlmr()

print("✅ TB-XLM-R fine-tuning complete")


In [ ]:

# 8. Reconstruct exact V14 validation probabilities/predictions
oldb=np.load(
    DATA_DIR/"old_bangla_snapshots.npz"
)
oldx=np.load(
    DATA_DIR/"old_xlmr_snapshots.npz"
)

def logit(p):
    p=np.clip(
        np.asarray(p,float),
        1e-6,1-1e-6
    )
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

v14v={}
v14t={}

v14v["is_toxic"]=(
    .875*v2v.is_toxic_prob.to_numpy(float)
    +.125*oldx["valid"][1:3,:,0].mean(0)
)
v14t["is_toxic"]=(
    .875*v2t.is_toxic_prob.to_numpy(float)
    +.125*oldx["test"][1:3,:,0].mean(0)
)

j=LI["implicit_toxicity"]
v14v["implicit_toxicity"]=sigmoid(
    .9*logit(v2v.implicit_toxicity_prob)
    +.1*logit(
        oldb["valid"].mean(0)[:,j]
    )
)
v14t["implicit_toxicity"]=sigmoid(
    .9*logit(v2t.implicit_toxicity_prob)
    +.1*logit(
        oldb["test"].mean(0)[:,j]
    )
)

v14v["sarcasm"]=v2v.sarcasm_prob.to_numpy(float)
v14t["sarcasm"]=v2t.sarcasm_prob.to_numpy(float)

j=LI["contains_profanity"]
v14v["contains_profanity"]=(
    .95*v2v.contains_profanity_prob.to_numpy(float)
    +.05*oldb["valid"][0,:,j]
)
v14t["contains_profanity"]=(
    .95*v2t.contains_profanity_prob.to_numpy(float)
    +.05*oldb["test"][0,:,j]
)

j=LI["threatening"]
v14v["threatening"]=(
    .95*v2v.threatening_prob.to_numpy(float)
    +.05*oldx["valid"][0,:,j]
)
v14t["threatening"]=(
    .95*v2t.threatening_prob.to_numpy(float)
    +.05*oldx["test"][0,:,j]
)

j=LI["identity_attack"]
v14v["identity_attack"]=(
    .70*v2v.identity_attack_prob.to_numpy(float)
    +.30*oldb["valid"][2,:,j]
)
v14t["identity_attack"]=(
    .70*v2t.identity_attack_prob.to_numpy(float)
    +.30*oldb["test"][2,:,j]
)

v14v["sexual_content"]=(
    v2v.sexual_content_prob.to_numpy(float)
)
v14t["sexual_content"]=(
    v2t.sexual_content_prob.to_numpy(float)
)

TH={
    "is_toxic":.6850110311061144,
    "implicit_toxicity":.1416,
    "sarcasm":.175,
    "contains_profanity":.245,
    "threatening":.16558668018085876,
    "identity_attack":.3814,
    "sexual_content":.245,
}

v14_pred=np.column_stack([
    (v14v[l]>=TH[l]).astype(int)
    for l in LABELS
])

print("✅ V14 reconstructed")


In [ ]:

# 9. Metric helpers + cross-script candidate construction
NTR=len(train)
VIDX=np.arange(
    NTR,NTR+len(valid)
)

yva=valid[LABELS].to_numpy(int)
valid_folds=combined.iloc[VIDX].fold.to_numpy(int)

def f1_pos(y,p):
    y=np.asarray(y,int)
    p=np.asarray(p,int)

    tp=((y==1)&(p==1)).sum()
    fp=((y==0)&(p==1)).sum()
    fn=((y==1)&(p==0)).sum()

    d=2*tp+fp+fn

    return float(2*tp/d) if d else 0.

def f1_macro(y,p):
    return float(np.mean([
        f1_pos(
            (y==c).astype(int),
            (p==c).astype(int)
        )
        for c in [0,1]
    ]))

def metric(label,y,p):
    return (
        f1_macro(y,p)
        if label=="is_toxic"
        else f1_pos(y,p)
    )

def official(y,p):
    ss=[
        metric(l,y[:,j],p[:,j])
        for j,l in enumerate(LABELS)
    ]

    return (
        float(
            .5*ss[0]
            +.5*np.mean(ss[1:])
        ),
        ss
    )

def topk_mask(prob,mask,k):
    idx=np.where(mask)[0]

    k=int(
        max(
            0,
            min(k,len(idx))
        )
    )

    out=np.zeros(
        len(prob),
        int
    )

    if k:
        order=np.argsort(
            -prob[idx]
        )
        out[idx[order[:k]]]=1

    return out

# Candidate probability families.
candidate_val={}
candidate_test={}

for j,label in enumerate(LABELS):
    candidate_val[label]={
        "e5":e5_oof[VIDX,j],
        "bge":bge_oof[VIDX,j],
        "tbxlmr":tb_oof[VIDX,j],
        "e5_tb":sigmoid(
            .5*logit(e5_oof[VIDX,j])
            +.5*logit(tb_oof[VIDX,j])
        ),
        "bge_tb":sigmoid(
            .5*logit(bge_oof[VIDX,j])
            +.5*logit(tb_oof[VIDX,j])
        ),
        "all3":sigmoid(
            (
                logit(e5_oof[VIDX,j])
                +logit(bge_oof[VIDX,j])
                +logit(tb_oof[VIDX,j])
            )/3
        ),
    }

    candidate_test[label]={
        "e5":e5_test[:,j],
        "bge":bge_test[:,j],
        "tbxlmr":tb_test[:,j],
        "e5_tb":sigmoid(
            .5*logit(e5_test[:,j])
            +.5*logit(tb_test[:,j])
        ),
        "bge_tb":sigmoid(
            .5*logit(bge_test[:,j])
            +.5*logit(tb_test[:,j])
        ),
        "all3":sigmoid(
            (
                logit(e5_test[:,j])
                +logit(bge_test[:,j])
                +logit(tb_test[:,j])
            )/3
        ),
    }

print("Candidate probability families ready")


In [ ]:

# 10. Conservative script-gated selection
#
# Native Bengali remains V14.
# We evaluate Roman and mixed rows separately.
# Prediction counts are anchored to labeled script-specific prevalence.

val_pred=v14_pred.copy()
test_pred=champ[LABELS].to_numpy(int).copy()

rows=[]
selected=[]

for j,label in enumerate(LABELS):
    base_full=metric(
        label,
        yva[:,j],
        v14_pred[:,j]
    )

    for script in ["roman","mixed"]:
        mv=valid.script.to_numpy()==script
        mt=test.script.to_numpy()==script

        if mv.sum()==0 or mt.sum()==0:
            continue

        base_script=metric(
            label,
            yva[mv,j],
            v14_pred[mv,j]
        )

        # Stable prior from all labeled rows of this script.
        lab_script=combined[
            combined.script==script
        ]

        prior=float(
            lab_script[label].mean()
        )

        # We allow only a very small prior menu.
        scales=[
            .80,
            1.00,
            1.20
        ]

        # For labels V14 predicts zero in a script,
        # the prior gives the specialist a chance to recover positives.
        best=None

        for source,pv in candidate_val[label].items():
            pt=candidate_test[label][source]

            for scale in scales:
                rate=prior*scale

                kv=int(round(
                    rate*mv.sum()
                ))

                kt=int(round(
                    rate*mt.sum()
                ))

                script_v=topk_mask(
                    pv,mv,kv
                )

                script_t=topk_mask(
                    pt,mt,kt
                )

                candidate_full=val_pred[:,j].copy()
                candidate_full[mv]=script_v[mv]

                full_score=metric(
                    label,
                    yva[:,j],
                    candidate_full
                )

                script_score=metric(
                    label,
                    yva[mv,j],
                    script_v[mv]
                )

                full_gain=(
                    full_score-base_full
                )

                script_gain=(
                    script_score-base_script
                )

                # Fold stability inside this script.
                wins=0
                losses=0
                ties=0

                for f in range(5):
                    fm=mv & (
                        valid_folds==f
                    )

                    if fm.sum()==0:
                        continue

                    # approximate same script rate
                    kk=int(round(
                        rate*fm.sum()
                    ))

                    cp=topk_mask(
                        pv,
                        fm,
                        kk
                    )[fm]

                    bp=v14_pred[
                        fm,j
                    ]

                    yy=yva[fm,j]

                    cs=metric(
                        label,yy,cp
                    )
                    bs=metric(
                        label,yy,bp
                    )

                    if cs>bs+1e-12:
                        wins+=1
                    elif cs<bs-1e-12:
                        losses+=1
                    else:
                        ties+=1

                # Script replacements must be materially useful.
                passed=(
                    script_gain>=.08
                    and full_gain>=.010
                    and wins>=2
                    and losses<=1
                )

                rec={
                    "label":label,
                    "script":script,
                    "source":source,
                    "scale":scale,
                    "prior":prior,
                    "validation_rows":int(mv.sum()),
                    "test_rows":int(mt.sum()),
                    "base_script_score":base_script,
                    "candidate_script_score":script_score,
                    "script_gain":script_gain,
                    "base_full_score":base_full,
                    "candidate_full_score":full_score,
                    "full_gain":full_gain,
                    "wins":wins,
                    "ties":ties,
                    "losses":losses,
                    "validation_positive_count":kv,
                    "test_positive_count":kt,
                    "passed":passed,
                }

                rows.append(rec)

                key=(
                    passed,
                    full_gain,
                    script_gain,
                    wins,
                    -losses
                )

                if (
                    best is None
                    or key>best["key"]
                ):
                    best={
                        "key":key,
                        "rec":rec,
                        "script_v":script_v,
                        "script_t":script_t,
                    }

        if best is not None and best["rec"]["passed"]:
            val_pred[mv,j]=best["script_v"][mv]
            test_pred[mt,j]=best["script_t"][mt]
            selected.append(
                best["rec"]
            )

search=pd.DataFrame(rows)
search.to_csv(
    RUN_DIR/"v35_script_candidate_search.csv",
    index=False
)

print("PASSED SCRIPT REPLACEMENTS:")
if selected:
    display(
        pd.DataFrame(
            selected
        ).round(5)
    )
else:
    print("None")


In [ ]:

# 11. Final LIVE validation score + submission

score,scores=official(
    yva,
    val_pred
)

baseline_score,baseline_scores=official(
    yva,
    v14_pred
)

report=pd.DataFrame({
    "label":LABELS,
    "v14_validation_score":baseline_scores,
    "v35_validation_score":scores,
    "gain":np.asarray(scores)-np.asarray(baseline_scores),
    "test_changes_vs_v14":[
        int(
            (
                test_pred[:,j]
                !=champ[l].to_numpy(int)
            ).sum()
        )
        for j,l in enumerate(LABELS)
    ],
    "v14_test_positive_rate":[
        float(
            champ[l].mean()
        )
        for l in LABELS
    ],
    "v35_test_positive_rate":
        test_pred.mean(axis=0),
})

report.to_csv(
    RUN_DIR/"v35_final_validation_report.csv",
    index=False
)

sub=champ.copy()

for j,l in enumerate(LABELS):
    sub[l]=test_pred[:,j].astype(int)

assert sub.ID.tolist()==test.ID.tolist()
assert sub.ID.tolist()==champ.ID.tolist()

assert sub.ID.head(4).tolist()==[
    "BICH_0003",
    "BICH_0006",
    "BICH_0015",
    "BICH_0016"
]

assert sub[LABELS].isin(
    [0,1]
).all().all()

sub.to_csv(
    RUN_DIR/"submission_v35_LIVE_crossscript.csv",
    index=False
)

summary={
    "version":
        "v35_live_crossscript_rescue",
    "protected_public_lb":0.586,
    "new_public_target":0.595,
    "v14_validation_score":
        float(baseline_score),
    "v35_validation_score":
        float(score),
    "validation_gain":
        float(score-baseline_score),
    "selected_script_replacements":
        selected,
    "total_test_changes":
        int(
            (
                test_pred
                !=champ[LABELS].to_numpy(int)
            ).sum()
        ),
    # Lower than our old 0.645 gate because V35 targets
    # a specific proven distribution weakness rather than
    # changing the whole submission.
    "SUBMIT_CANDIDATE":bool(
        score-baseline_score>=.008
        and len(selected)>=1
    )
}

(RUN_DIR/"v35_summary.json").write_text(
    json.dumps(
        summary,
        indent=2
    )
)

print(
    json.dumps(
        summary,
        indent=2
    )
)

display(
    report.round(5)
)

if summary["SUBMIT_CANDIDATE"]:
    print(
        "\n✅ V35 passed the cross-script gate. "
        "Send v35_results.zip before submitting."
    )
else:
    print(
        "\n❌ V35 did not pass. "
        "Do NOT submit; send results ZIP for analysis."
    )


In [ ]:

# 12. Package V35 results

np.savez_compressed(
    RUN_DIR/"v35_new_probabilities.npz",
    e5_oof=e5_oof,
    e5_test=e5_test,
    bge_oof=bge_oof,
    bge_test=bge_test,
    tbxlmr_oof=tb_oof,
    tbxlmr_test=tb_test,
)

zip_path=Path(
    "/content/v35_results.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(
                p,
                arcname=p.name
            )

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
